# Matchup Analysis — Do Style Clusters Predict Model Errors?

Hypothesis: certain team style archetypes systematically beat or lose to other archetypes
in ways that our RAPM-based model doesn't capture.

Method:
1. Run full-season backtest to get predicted vs actual margins
2. Assign home/away team clusters for each game
3. Compute residuals (actual - predicted) cross-tabulated by (home_cluster, away_cluster)
4. Test for statistically significant cluster-pair biases using bootstrap CI

In [ ]:
import sys
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats

sys.path.insert(0, '..')
from config import DB_PATH, NBA_TEAM_MAP, INITIAL_SEASON
from downstream.backtest import run_backtest

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print("Setup complete")

## Step 1: Load Cluster Assignments

In [ ]:
with open('data/cluster_assignments.json', 'r') as f:
    cluster_data = json.load(f)

cluster_by_team_id = cluster_data['cluster_by_team_id']
cluster_names = cluster_data['cluster_names']  # dict {str(k): name}
K = cluster_data['k']
abbrev = {v['abbrev']: k for k, v in NBA_TEAM_MAP.items()}  # abbrev -> team_id
team_abbrev = {k: v['abbrev'] for k, v in NBA_TEAM_MAP.items()}  # team_id -> abbrev

print(f"Loaded {K} clusters")
for c_id, c_name in cluster_names.items():
    teams_in_c = [team_abbrev[tid] for tid, cid in cluster_by_team_id.items() if str(cid) == str(c_id)]
    print(f"  {c_name}: {sorted(teams_in_c)}")

## Step 2: Run Full-Season Backtest

In [ ]:
# This may take 5-10 minutes — runs prediction model on all scored games
# Uses possession history before each game date (no oracle/lookahead bias)
print("Running full-season backtest...")
print("(This takes several minutes — runs Monte Carlo simulations for each game)")

results = run_backtest(
    db_path=DB_PATH,
    season=INITIAL_SEASON,
    n_simulations=200,  # reduced for speed in analysis; production uses 500
)

df_results = pd.DataFrame(results)
print(f"\nBacktest complete: {len(df_results)} games")
print(df_results[['game_date','home_team_id','away_team_id','predicted_spread','actual_margin','error']].head(10).to_string())

In [ ]:
# Add residual column (actual - predicted, positive = home outperformed)
df_results['residual'] = df_results['actual_margin'] - df_results['predicted_spread']

# Add cluster assignments
df_results['home_cluster'] = df_results['home_team_id'].map(cluster_by_team_id)
df_results['away_cluster'] = df_results['away_team_id'].map(cluster_by_team_id)

# Drop games where cluster assignment is missing
n_before = len(df_results)
df_results = df_results.dropna(subset=['home_cluster', 'away_cluster'])
df_results['home_cluster'] = df_results['home_cluster'].astype(int)
df_results['away_cluster'] = df_results['away_cluster'].astype(int)
print(f"Games with cluster assignments: {len(df_results)} / {n_before}")

# Overall residual stats (should be near 0 — model is calibrated)
print(f"\nOverall residual stats:")
print(f"  Mean residual: {df_results['residual'].mean():.2f} pts (ideal: 0)")
print(f"  Std residual: {df_results['residual'].std():.2f} pts")

## Step 3: Cross-Tab by Cluster Pair

In [ ]:
# For each (home_cluster, away_cluster) pair, compute mean residual, count, and std
records = []
for hc in range(K):
    for ac in range(K):
        subset = df_results[
            (df_results['home_cluster'] == hc) &
            (df_results['away_cluster'] == ac)
        ]
        if len(subset) > 0:
            records.append({
                'home_cluster': hc,
                'away_cluster': ac,
                'n_games': len(subset),
                'mean_residual': subset['residual'].mean(),
                'std_residual': subset['residual'].std(),
                'mean_abs_error': subset['abs_error'].mean(),
            })

df_pairs = pd.DataFrame(records)

# Pivot to K×K matrix
residual_matrix = df_pairs.pivot(index='home_cluster', columns='away_cluster', values='mean_residual')
count_matrix = df_pairs.pivot(index='home_cluster', columns='away_cluster', values='n_games')

# Print raw results
print("Cluster-pair mean residuals (actual - predicted, home team perspective):")
print(f"{'':>20}", end='')
for ac in range(K):
    print(f"  Away C{ac}", end='')
print()
for hc in range(K):
    print(f"Home C{hc}: ", end='')
    for ac in range(K):
        val = residual_matrix.loc[hc, ac] if hc in residual_matrix.index and ac in residual_matrix.columns else float('nan')
        n = count_matrix.loc[hc, ac] if hc in count_matrix.index and ac in count_matrix.columns else 0
        print(f"  {val:+.1f}({n:>2})", end='')
    print()

In [ ]:
# Bootstrap confidence intervals for each pair
def bootstrap_ci(data, n_boot=1000, ci=0.95):
    """Returns (lower, upper) CI for the mean."""
    if len(data) < 3:
        return (float('nan'), float('nan'))
    boot_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    lower = np.percentile(boot_means, (1-ci)/2 * 100)
    upper = np.percentile(boot_means, (1+ci)/2 * 100)
    return (lower, upper)

# Add CI columns
np.random.seed(42)
ci_records = []
for _, row in df_pairs.iterrows():
    hc, ac = int(row['home_cluster']), int(row['away_cluster'])
    subset = df_results[
        (df_results['home_cluster'] == hc) &
        (df_results['away_cluster'] == ac)
    ]['residual'].values
    lo, hi = bootstrap_ci(subset)
    ci_records.append({
        'home_cluster': hc,
        'away_cluster': ac,
        'ci_lower': lo,
        'ci_upper': hi,
        'significant': (lo > 0) or (hi < 0),  # CI excludes 0
    })
df_ci = pd.DataFrame(ci_records)
df_pairs = df_pairs.merge(df_ci, on=['home_cluster', 'away_cluster'])

# Print significant pairs
print("Statistically significant cluster pairs (bootstrap 95% CI excludes 0, n >= 10):")
sig = df_pairs[(df_pairs['significant']) & (df_pairs['n_games'] >= 10)].sort_values('mean_residual')
if len(sig) == 0:
    print("  No significant pairs found at n>=10 threshold.")
    sig_any = df_pairs[df_pairs['significant']].sort_values('mean_residual')
    print(f"  (Significant at any n: {len(sig_any)} pairs)")
else:
    for _, row in sig.iterrows():
        hc, ac = int(row['home_cluster']), int(row['away_cluster'])
        h_name = cluster_names.get(str(hc), f'C{hc}')
        a_name = cluster_names.get(str(ac), f'C{ac}')
        print(f"  Home {h_name} vs Away {a_name}:")
        print(f"    n={int(row['n_games'])}, mean={row['mean_residual']:+.2f}, 95% CI=[{row['ci_lower']:+.2f}, {row['ci_upper']:+.2f}]")

## Step 4: Heatmap Visualization

In [ ]:
# Build cluster name labels
c_labels = [cluster_names.get(str(c), f'C{c}') for c in range(K)]
# Shorten for display
c_short = [n.split('\u2014')[1].strip() if '\u2014' in n else n for n in c_labels]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: mean residual heatmap
residual_arr = residual_matrix.values
im = axes[0].imshow(residual_arr, cmap='RdBu_r', vmin=-8, vmax=8, aspect='auto')
axes[0].set_xticks(range(K)); axes[0].set_yticks(range(K))
axes[0].set_xticklabels([f'C{i}' for i in range(K)], fontsize=9)
axes[0].set_yticklabels([f'C{i}' for i in range(K)], fontsize=9)
axes[0].set_xlabel('Away Cluster')
axes[0].set_ylabel('Home Cluster')
axes[0].set_title('Mean Residual (actual - predicted)\nPositive = home outperformed model')
# Annotate cells with value (n)
for i in range(K):
    for j in range(K):
        val = residual_matrix.iloc[i, j] if i < len(residual_matrix) and j < residual_matrix.shape[1] else float('nan')
        n_val = count_matrix.iloc[i, j] if i < len(count_matrix) and j < count_matrix.shape[1] else 0
        if not np.isnan(val):
            sig_marker = '*' if df_pairs[(df_pairs['home_cluster']==i) & (df_pairs['away_cluster']==j)]['significant'].values[0] else ''
            axes[0].text(j, i, f'{val:+.1f}{sig_marker}\n(n={int(n_val)})', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=axes[0])

# Right: game count heatmap
count_arr = count_matrix.values
im2 = axes[1].imshow(count_arr, cmap='Blues', aspect='auto')
axes[1].set_xticks(range(K)); axes[1].set_yticks(range(K))
axes[1].set_xticklabels([f'C{i}' for i in range(K)], fontsize=9)
axes[1].set_yticklabels([f'C{i}' for i in range(K)], fontsize=9)
axes[1].set_xlabel('Away Cluster')
axes[1].set_ylabel('Home Cluster')
axes[1].set_title('Game Count per Cluster Pair')
for i in range(K):
    for j in range(K):
        n_val = count_matrix.iloc[i, j] if i < len(count_matrix) and j < count_matrix.shape[1] else 0
        if not np.isnan(n_val):
            axes[1].text(j, i, str(int(n_val)), ha='center', va='center', fontsize=10)
plt.colorbar(im2, ax=axes[1])

plt.suptitle('Cluster-Pair Matchup Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('data/matchup_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print("* = 95% bootstrap CI excludes 0 (statistically significant)")

## Step 5: Deep Dive — SAS vs OKC (Wemby Hypothesis)

In [ ]:
# Find SAS and OKC team IDs
sas_id = next(k for k, v in NBA_TEAM_MAP.items() if v['abbrev'] == 'SAS')
okc_id = next(k for k, v in NBA_TEAM_MAP.items() if v['abbrev'] == 'OKC')

sas_cluster = cluster_by_team_id.get(sas_id)
okc_cluster = cluster_by_team_id.get(okc_id)

print(f"SAS cluster: {sas_cluster} \u2014 {cluster_names.get(str(sas_cluster), 'Unknown')}")
print(f"OKC cluster: {okc_cluster} \u2014 {cluster_names.get(str(okc_cluster), 'Unknown')}")

# All H2H games SAS vs OKC this season
sas_okc = df_results[
    ((df_results['home_team_id'] == sas_id) & (df_results['away_team_id'] == okc_id)) |
    ((df_results['home_team_id'] == okc_id) & (df_results['away_team_id'] == sas_id))
].copy()

print(f"\nSAS vs OKC this season: {len(sas_okc)} games")
if len(sas_okc) > 0:
    sas_okc['home_abbrev'] = sas_okc['home_team_id'].map(team_abbrev)
    sas_okc['away_abbrev'] = sas_okc['away_team_id'].map(team_abbrev)
    for _, row in sas_okc.iterrows():
        print(f"  {row['away_abbrev']} @ {row['home_abbrev']} ({row['game_date']}): "
              f"pred={row['predicted_spread']:+.1f}, actual={row['actual_margin']:+.0f}, "
              f"residual={row['residual']:+.1f}")
else:
    print("  No H2H games found yet this season")

In [ ]:
# Broader cluster analysis: how does OKC's cluster perform vs SAS's cluster?
if sas_cluster is not None and okc_cluster is not None:
    # Games where OKC cluster is home, SAS cluster is away
    okc_home_vs_sas = df_results[
        (df_results['home_cluster'] == okc_cluster) &
        (df_results['away_cluster'] == sas_cluster)
    ]
    # Games where SAS cluster is home, OKC cluster is away
    sas_home_vs_okc = df_results[
        (df_results['home_cluster'] == sas_cluster) &
        (df_results['away_cluster'] == okc_cluster)
    ]

    print(f"OKC-type (C{okc_cluster}) hosting SAS-type (C{sas_cluster}):")
    print(f"  n={len(okc_home_vs_sas)}, mean residual={okc_home_vs_sas['residual'].mean():+.2f}")

    print(f"\nSAS-type (C{sas_cluster}) hosting OKC-type (C{okc_cluster}):")
    print(f"  n={len(sas_home_vs_okc)}, mean residual={sas_home_vs_okc['residual'].mean():+.2f}")

## Step 6: Significant Pairs — Team-Level Breakdown

In [ ]:
# For each significant pair, list all games with team abbreviations
sig_pairs = df_pairs[(df_pairs['significant']) & (df_pairs['n_games'] >= 8)]

if len(sig_pairs) == 0:
    # Lower threshold if no significant pairs at n>=8
    sig_pairs = df_pairs[df_pairs['significant']].nlargest(4, 'n_games')
    print("No pairs with n>=8 significant; showing largest-n significant pairs:")

for _, row in sig_pairs.sort_values('mean_residual').iterrows():
    hc, ac = int(row['home_cluster']), int(row['away_cluster'])
    h_name = cluster_names.get(str(hc), f'C{hc}')
    a_name = cluster_names.get(str(ac), f'C{ac}')

    subset = df_results[
        (df_results['home_cluster'] == hc) &
        (df_results['away_cluster'] == ac)
    ].copy()
    subset['home_abbrev'] = subset['home_team_id'].map(team_abbrev)
    subset['away_abbrev'] = subset['away_team_id'].map(team_abbrev)

    print(f"\n{'='*60}")
    print(f"Home: {h_name} vs Away: {a_name}")
    print(f"n={int(row['n_games'])}, mean residual={row['mean_residual']:+.2f}, "
          f"95% CI=[{row['ci_lower']:+.2f}, {row['ci_upper']:+.2f}]")
    print(f"Interpretation: {'home team outperforms model' if row['mean_residual'] > 0 else 'home team underperforms model'}")
    print(f"Games:")
    for _, g in subset.sort_values('residual').iterrows():
        print(f"  {g['away_abbrev']} @ {g['home_abbrev']} ({g['game_date']}): "
              f"pred={g['predicted_spread']:+.1f}, actual={g['actual_margin']:+.0f}, "
              f"residual={g['residual']:+.1f}")

## Summary and Findings

In [ ]:
# Summary stats
print("="*60)
print("MATCHUP ANALYSIS SUMMARY")
print("="*60)
print(f"\nTotal games analyzed: {len(df_results)}")
print(f"Overall mean residual: {df_results['residual'].mean():+.2f} pts (bias check \u2014 ideal: 0)")
print(f"Overall std residual: {df_results['residual'].std():.2f} pts")
print(f"\nCluster pairs tested: {len(df_pairs)}")
print(f"Significant pairs (95% CI, any n): {df_pairs['significant'].sum()}")
print(f"Significant pairs (95% CI, n>=10): {((df_pairs['significant']) & (df_pairs['n_games']>=10)).sum()}")
print("\nLargest positive residuals (home outperforms model):")
print(df_pairs.nlargest(3, 'mean_residual')[['home_cluster','away_cluster','mean_residual','n_games','significant']].to_string())
print("\nLargest negative residuals (home underperforms model):")
print(df_pairs.nsmallest(3, 'mean_residual')[['home_cluster','away_cluster','mean_residual','n_games','significant']].to_string())

In [ ]:
# Save results
df_results.to_csv('data/matchup_residuals.csv', index=False)
df_pairs.to_csv('data/cluster_pair_analysis.csv', index=False)
print("Saved data/matchup_residuals.csv")
print("Saved data/cluster_pair_analysis.csv")

## Next Steps

Based on these findings:
1. If any cluster pairs show |mean residual| > 4 pts with n >= 20 and significant CI:
   \u2192 Add cluster-pair dummy to `downstream/calibration.py` and refit
   \u2192 Backtest before/after to confirm improvement

2. Examine the specific teams in significant cluster pairs \u2014 do they make NBA sense?
   (Ask David to validate: "Does this style matchup edge make intuitive sense?")

3. Look for the SAS/Wemby pattern:
   \u2192 SAS hosting OKC: does OKC underperform (negative residual)?
   \u2192 The "rim protection neutralizes ball-screen heavy offense" hypothesis

4. Track these edges across the remainder of the 2025-26 season to see if they persist